In [22]:
import os
os.environ["OPENAI_API_KEY"] = "ollama"
os.environ["OPENAI_BASE_URL"] = "http://localhost:11434/v1"

from pathlib import Path
import json
import re
import tempfile
from typing import Any, Dict, List

import pandas as pd
from openai import OpenAI
from lida import Manager, llm


In [23]:
import seaborn as sns


In [24]:
def flatten_json(obj, parent="", sep="_"):
    out = {}
    if isinstance(obj, dict):
        for k, v in obj.items():
            key = f"{parent}{sep}{k}" if parent else k
            if isinstance(v, dict):
                out.update(flatten_json(v, key, sep))
            else:
                out[key] = v
    return out


def load_any_df(path: str, sample_rows=5000):
    p = Path(path)
    ext = p.suffix.lower()

    if ext in [".csv", ".tsv"]:
        return pd.read_csv(p, nrows=sample_rows)

    if ext in [".log", ".ndjson", ".jsonl"]:
        rows = []
        with open(p, encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i >= sample_rows:
                    break
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                    rows.append(flatten_json(obj))
                except:
                    continue
        return pd.DataFrame(rows)

    return pd.read_csv(p, nrows=sample_rows)


In [25]:
def sanitize_mapping(mapping: dict, df: pd.DataFrame):
    cols = set(df.columns)

    if not isinstance(mapping, dict):
        mapping = {}

    mapping.setdefault("x", None)
    mapping.setdefault("y", None)
    mapping.setdefault("chart", "auto")
    if mapping.get("filter") is None:
        mapping["filter"] = []

    for k in ["x","y"]:
        if mapping.get(k) not in cols:
            mapping[k] = None

    valid = []
    for f in (mapping.get("filter") or []):
        if isinstance(f, dict) and f.get("column") in cols:
            valid.append(f)
    mapping["filter"] = valid

    return mapping


def apply_filters(df, mapping):
    out = df.copy()
    for f in mapping.get("filter", []):
        col = f["column"]
        op = f["op"]
        val = f["value"]

        if col not in out.columns:
            continue

        s = out[col]

        if s.dropna().isin([True, False]).all():
            if op == "==":
                out = out[s == bool(val)]
            elif op == "!=":
                out = out[s != bool(val)]
            continue

        if op == "==":
            out = out[s == val]
        elif op == "!=":
            out = out[s != val]

    return out


In [26]:
def llm_map_columns(prompt: str, df: pd.DataFrame):
    schema = list(df.columns)

    client = OpenAI(
        api_key="ollama",
        base_url="http://localhost:11434/v1"
    )

    system = """Return ONLY valid JSON:
{
 "x": string|null,
 "y": string|null,
 "chart": "bar"|"pie"|"scatter"|"line"|"auto",
 "filter": []
}
Use only column names from the schema."""

    msg = f"Schema: {schema}\nUser request: {prompt}"

    resp = client.chat.completions.create(
        model="mistral:7b",
        messages=[
            {"role":"system","content":system},
            {"role":"user","content":msg}
        ],
        temperature=0.1
    )

    text = resp.choices[0].message.content.strip()

    try:
        mapping = json.loads(text)
    except:
        mapping = {"x":None,"y":None,"chart":"auto","filter":[]}

    return mapping


In [27]:
def prepare_count_df(df, group_col):
    out = df[group_col].value_counts().reset_index()
    out.columns = [group_col, "count"]
    return out


In [33]:
def run_lida_clean(file, prompt, out="out.png"):

    df = load_any_df(file)

    mapping = llm_map_columns(prompt, df)
    mapping = sanitize_mapping(mapping, df)

    print("\nMAPPING:", mapping)

    df2 = apply_filters(df, mapping)

    # auto detect "nombre"
    if "nombre" in prompt.lower() or "count" in prompt.lower():
        if mapping["x"] in df2.columns:
            df_plot = prepare_count_df(df2, mapping["x"])
            mapping["y"] = "count"
            mapping["chart"] = "bar"
        else:
            df_plot = df2
    else:
        df_plot = df2

    print("\nDF_PLOT HEAD:")
    print(df_plot.head())

    with tempfile.NamedTemporaryFile("w", suffix=".csv", delete=False) as tmp:
        df_plot.to_csv(tmp.name, index=False)
        tmp_path = tmp.name

    lida_mgr = Manager(text_gen=llm("openai", model="mistral:7b"))
    summary = lida_mgr.summarize(tmp_path)

    goal = {
        "question": prompt,
        "visualization": f"Create a {mapping['chart']} chart using matplotlib. Use x={mapping['x']} and y={mapping['y']}.",
        "rationale": "Mapping from prompt to columns."
    }

    charts = lida_mgr.visualize(summary=summary, goal=goal, library="matplotlib")

    if charts:

        chart_obj = charts[0]

        raster = getattr(chart_obj, "raster", None)

        if isinstance(raster, (bytes, bytearray)):
            with open(out, "wb") as f:
              f.write(raster)

        elif isinstance(raster, str):
         import base64
         if raster.startswith("data:image"):
             raster = raster.split(",", 1)[1]
         with open(out, "wb") as f:
            f.write(base64.b64decode(raster))

        else:
            print("No raster found in LIDA response.")

        print("\nGraph generated:", out)

    else:
        print("No chart generated.")


In [ ]:
run_lida_clean(
    file="data/lidata.log",
    prompt="Bar chart du nombre d'entités par EntityType",
    
)

TypeError: run_lida_clean() got an unexpected keyword argument 'model'

In [35]:

from __future__ import annotations

import json, re
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
import matplotlib.pyplot as plt
from openai import OpenAI


# ----------------------------
# IO: csv / json / ndjson / .log(ndjson)
# ----------------------------

def _flatten_json(obj: Any, parent: str = "", sep: str = ".", max_depth: int = 12, _d: int = 0) -> Dict[str, Any]:
    out: Dict[str, Any] = {}
    if _d > max_depth:
        out[parent or "value"] = json.dumps(obj, ensure_ascii=False)
        return out
    if isinstance(obj, dict):
        for k, v in obj.items():
            key = f"{parent}{sep}{k}" if parent else str(k)
            if isinstance(v, dict):
                out.update(_flatten_json(v, key, sep, max_depth, _d + 1))
            elif isinstance(v, list):
                out[key] = json.dumps(v, ensure_ascii=False)
            else:
                out[key] = v
        return out
    if isinstance(obj, list):
        out[parent or "value"] = json.dumps(obj, ensure_ascii=False)
        return out
    out[parent or "value"] = obj
    return out

def _read_ndjson_rows(path: Path, limit: int = 5000) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= limit:
                break
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            if isinstance(obj, dict):
                rows.append(_flatten_json(obj))
    return rows

def load_any_df(path: str, sample_rows: int = 5000) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(path)
    ext = p.suffix.lower()

    if ext in [".csv", ".tsv"]:
        sep = "\t" if ext == ".tsv" else ","
        try:
            return pd.read_csv(p, sep=sep, nrows=sample_rows, low_memory=False)
        except Exception:
            return pd.read_csv(p, sep=None, engine="python", nrows=sample_rows, low_memory=False)

    if ext in [".json", ".ndjson", ".jsonl", ".log"]:
        # json array OR ndjson lines
        txt = p.read_text(encoding="utf-8", errors="ignore").strip()
        if txt.startswith("["):
            data = json.loads(txt)
            if not isinstance(data, list):
                data = [data]
            data = data[:sample_rows]
            rows = [_flatten_json(x) if isinstance(x, dict) else {"value": json.dumps(x, ensure_ascii=False)} for x in data]
            return pd.DataFrame(rows)
        return pd.DataFrame(_read_ndjson_rows(p, limit=sample_rows))

    # fallback
    return pd.read_csv(p, nrows=sample_rows, low_memory=False)


# ----------------------------
# Type coercion (important)
# ----------------------------

def coerce_basic_types(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # bool-like strings -> bool
    for col in out.columns:
        if out[col].dtype == "object":
            s = out[col].dropna().astype(str).str.strip().str.lower()
            if s.empty:
                continue
            # if mostly true/false
            tf_ratio = s.isin(["true", "false", "0", "1"]).mean()
            if tf_ratio >= 0.95:
                out[col] = (
                    out[col]
                    .astype(str).str.strip().str.lower()
                    .map({"true": True, "false": False, "1": True, "0": False})
                )

    # numeric-like strings -> numeric
    for col in out.columns:
        if out[col].dtype == "object":
            s = out[col].dropna().astype(str).head(80)
            if s.empty:
                continue
            numeric_like = s.str.match(r"^\s*-?\d+(\.\d+)?\s*$").mean()
            if numeric_like >= 0.85:
                out[col] = pd.to_numeric(out[col], errors="coerce")

    return out


# ----------------------------
# LLM mapping (Ollama OpenAI-compatible)
# ----------------------------

MAPPER_SYSTEM = """Return ONLY one JSON object. No markdown. No extra text.

Schema:
{
  "x": string|null,
  "y": string|null,
  "color": string|null,
  "time": string|null,
  "chart": "bar"|"pie"|"scatter"|"line"|"hist"|"auto",
  "filter": [{"column": string, "op": "=="|"!="|">"|">="|"<"|"<="|"in"|"contains", "value": any}]
}

Rules:
- Use ONLY columns that exist in the provided COLUMNS list.
- For 'bar chart du nombre ... par X': set chart="bar", x="X", y="count".
- For pie: chart="pie", x=category column, y="count".
- If unsure, set fields null and chart="auto".
"""

def _parse_json_object_loose(text: str) -> dict:
    t = (text or "").strip()
    t = re.sub(r"^```(?:json)?\s*", "", t, flags=re.IGNORECASE).strip()
    t = re.sub(r"\s*```$", "", t).strip()

    try:
        obj = json.loads(t)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass

    start, end = t.find("{"), t.rfind("}")
    if start != -1 and end != -1 and end > start:
        candidate = t[start:end+1]
        candidate = candidate.replace("\u201c", '"').replace("\u201d", '"')
        obj = json.loads(candidate)
        if isinstance(obj, dict):
            return obj

    raise ValueError("Could not extract JSON object")

def llm_map_columns(prompt: str, columns: List[str], model: str = "mistral:7b") -> Dict[str, Any]:
    # Ollama OpenAI-compatible endpoint
    client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")

    msg = f"COLUMNS:\n{json.dumps(columns, ensure_ascii=False)}\n\nUSER_PROMPT:\n{prompt}\n\nReturn JSON only."

    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": MAPPER_SYSTEM},
            {"role": "user", "content": msg},
        ],
        temperature=0.1,
    )
    content = (resp.choices[0].message.content or "").strip()
    return _parse_json_object_loose(content)


# ----------------------------
# Sanitize + fallback (key to robustness)
# ----------------------------

def sanitize_mapping(mapping: Dict[str, Any], df: pd.DataFrame) -> Dict[str, Any]:
    cols = set(df.columns)

    # normalize keys
    for k in ["x", "y", "color", "time", "chart", "filter"]:
        mapping.setdefault(k, None)

    # fix chart casing
    if isinstance(mapping.get("chart"), str):
        mapping["chart"] = mapping["chart"].strip().lower()
    if mapping["chart"] not in ["bar", "pie", "scatter", "line", "hist", "auto"]:
        mapping["chart"] = "auto"

    # direct fields
    for k in ["x", "y", "color", "time"]:
        v = mapping.get(k)
        if isinstance(v, str):
            v = v.strip()
            mapping[k] = v if v in cols or v in ["count"] else None
        else:
            mapping[k] = None

    # filters
    flt = mapping.get("filter")
    if not isinstance(flt, list):
        flt = []
    valid = []
    for f in flt:
        if (
            isinstance(f, dict)
            and f.get("column") in cols
            and f.get("op") in ["==","!=",">",">=","<","<=","in","contains"]
        ):
            valid.append(f)
    mapping["filter"] = valid

    return mapping

def fallback_mapping(prompt: str, df: pd.DataFrame, mapping: Dict[str, Any]) -> Dict[str, Any]:
    p = prompt.lower()
    cols = list(df.columns)

    # If prompt mentions a column explicitly (case-insensitive), prefer it
    for c in cols:
        if c.lower() in p:
            if mapping.get("x") is None:
                mapping["x"] = c
            break

    # Heuristic for "nombre / count / par X"
    if ("bar" in p or "hist" in p or "nombre" in p or "count" in p) and mapping.get("x"):
        mapping["chart"] = "bar"
        mapping["y"] = "count"

    # auto fallback if still empty
    if mapping.get("chart") == "auto":
        # common intents
        if "pie" in p or "camembert" in p:
            mapping["chart"] = "pie"
        elif "scatter" in p or "nuage" in p or "position" in p:
            mapping["chart"] = "scatter"
        elif "courbe" in p or "line" in p or "evolution" in p:
            mapping["chart"] = "line"

    # still no x: choose best categorical
    if mapping.get("x") is None:
        cat = [c for c in cols if df[c].dtype == "object" and df[c].nunique(dropna=True) > 1]
        if cat:
            mapping["x"] = cat[0]

    # line: choose a time column if exists
    if mapping.get("chart") == "line" and mapping.get("time") is None:
        for cand in ["SimTime", "@timestamp", "timestamp", "time", "date"]:
            if cand in df.columns:
                mapping["time"] = cand
                break

    return mapping


# ----------------------------
# Apply filters (safe)
# ----------------------------

def apply_filters(df: pd.DataFrame, mapping: Dict[str, Any]) -> pd.DataFrame:
    out = df
    for flt in (mapping.get("filter") or []):
        col = flt.get("column")
        op = flt.get("op")
        val = flt.get("value")
        if col not in out.columns:
            continue
        s = out[col]

        if op == "==":
            out = out[s == val]
        elif op == "!=":
            out = out[s != val]
        elif op == ">":
            out = out[pd.to_numeric(s, errors="coerce") > float(val)]
        elif op == ">=":
            out = out[pd.to_numeric(s, errors="coerce") >= float(val)]
        elif op == "<":
            out = out[pd.to_numeric(s, errors="coerce") < float(val)]
        elif op == "<=":
            out = out[pd.to_numeric(s, errors="coerce") <= float(val)]
        elif op == "contains":
            out = out[s.astype(str).str.contains(str(val), na=False)]
        elif op == "in" and isinstance(val, list):
            out = out[s.isin(val)]
    return out


# ----------------------------
# Deterministic plotting engine (matplotlib only)
# ----------------------------

def plot_bar_count(df: pd.DataFrame, xcol: str, out: str, title: str):
    counts = df[xcol].astype(str).value_counts().reset_index()
    counts.columns = [xcol, "count"]

    fig, ax = plt.subplots()
    ax.bar(counts[xcol], counts["count"])
    ax.set_xlabel(xcol)
    ax.set_ylabel("count")
    ax.set_title(title)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(out)
    plt.close()

def plot_pie_count(df: pd.DataFrame, xcol: str, out: str, title: str, max_slices: int = 12):
    counts = df[xcol].astype(str).value_counts()
    if len(counts) > max_slices:
        top = counts.iloc[:max_slices-1]
        other = counts.iloc[max_slices-1:].sum()
        counts = pd.concat([top, pd.Series({"Other": other})])

    fig, ax = plt.subplots()
    ax.pie(counts.values, labels=counts.index.tolist(), autopct="%1.1f%%", startangle=90)
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(out)
    plt.close()

def plot_scatter(df: pd.DataFrame, xcol: str, ycol: str, out: str, title: str, color: Optional[str] = None):
    fig, ax = plt.subplots()
    if color and color in df.columns:
        # simple grouping without seaborn
        for k, g in df.groupby(color):
            ax.scatter(pd.to_numeric(g[xcol], errors="coerce"), pd.to_numeric(g[ycol], errors="coerce"), label=str(k))
        ax.legend()
    else:
        ax.scatter(pd.to_numeric(df[xcol], errors="coerce"), pd.to_numeric(df[ycol], errors="coerce"))

    ax.set_xlabel(xcol)
    ax.set_ylabel(ycol)
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(out)
    plt.close()

def plot_line(df: pd.DataFrame, timecol: str, ycol: str, out: str, title: str):
    d = df[[timecol, ycol]].copy()
    d[timecol] = pd.to_numeric(d[timecol], errors="coerce")
    d[ycol] = pd.to_numeric(d[ycol], errors="coerce")
    d = d.dropna().sort_values(timecol)

    fig, ax = plt.subplots()
    ax.plot(d[timecol], d[ycol])
    ax.set_xlabel(timecol)
    ax.set_ylabel(ycol)
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(out)
    plt.close()


# ----------------------------
# ✅ run_lida_clean (propre)
# ----------------------------

def run_lida_clean(
    file: str,
    prompt: str,
    out: str = "out.png",
    sample_rows: int = 5000,
    model: str = "mistral:7b",
):
    df = load_any_df(file, sample_rows=sample_rows)
    if df.empty:
        raise RuntimeError("Empty dataframe after parsing.")
    df = coerce_basic_types(df)

    # LLM mapping (columns only, not plotting code)
    cols = list(df.columns)
    mapping = llm_map_columns(prompt, cols, model=model)
    mapping = sanitize_mapping(mapping, df)
    mapping = fallback_mapping(prompt, df, mapping)

    print("MAPPING:", mapping)

    # apply filters
    df2 = apply_filters(df, mapping)
    if df2.empty:
        raise RuntimeError("No rows after filters.")

    chart = mapping.get("chart") or "auto"
    x = mapping.get("x")
    y = mapping.get("y")
    color = mapping.get("color")
    timecol = mapping.get("time")

    # Deterministic routing
    if chart == "bar":
        # bar count by x
        if x is None:
            raise RuntimeError("Bar chart needs x column.")
        plot_bar_count(df2, x, out, prompt)
        print("✅ Graph generated:", out)
        return

    if chart == "pie":
        if x is None:
            raise RuntimeError("Pie chart needs x column (category).")
        plot_pie_count(df2, x, out, prompt)
        print("✅ Graph generated:", out)
        return

    if chart == "scatter":
        # needs x and y; try auto-detect common spatial columns if missing
        if x is None or y is None:
            spatial_x = [c for c in df2.columns if c.endswith("WorldLocation_x") or c.endswith(".x")]
            spatial_y = [c for c in df2.columns if c.endswith("WorldLocation_y") or c.endswith(".y")]
            if spatial_x and spatial_y:
                x = spatial_x[0]
                y = spatial_y[0]
            else:
                raise RuntimeError("Scatter needs x and y columns (or detectable spatial columns).")
        plot_scatter(df2, x, y, out, prompt, color=color if isinstance(color, str) else None)
        print("✅ Graph generated:", out)
        return

    if chart == "line":
        # needs time and y
        if timecol is None:
            for cand in ["SimTime", "@timestamp", "timestamp", "time", "date"]:
                if cand in df2.columns:
                    timecol = cand
                    break
        if timecol is None or y is None:
            raise RuntimeError("Line chart needs time and y columns.")
        plot_line(df2, timecol, y, out, prompt)
        print("✅ Graph generated:", out)
        return

    # auto fallback: if prompt suggests count-by-column
    p = prompt.lower()
    if x and ("par" in p or "nombre" in p or "count" in p):
        plot_bar_count(df2, x, out, prompt)
        print("✅ Graph generated (auto->bar):", out)
        return

    raise RuntimeError("Unsupported/unclear chart request. (Try: bar/pie/scatter/line)")


In [36]:
run_lida_clean(
    file="data/lidata.log",
    prompt="Bar chart du nombre d'entités par EntityType",
    out="out2.png",
    model="mistral:7b"
)


MAPPING: {'chart': 'bar', 'x': 'EntityType', 'y': 'count', 'color': None, 'time': None, 'filter': []}
✅ Graph generated: out2.png
